<a href="https://colab.research.google.com/github/abhishekjoshi13/OptimalSafety-Routing/blob/main/OptimalSafety_Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import random
np.random.seed(42)
numRows = 50000
zones = ['Commercial', 'Residential', 'Industrial', 'Outskirts']
times = ['Morning', 'Afternoon', 'Evening', 'LateNight']
data = {
    'zone': [], 'timeOfDay': [], 'luxLevel': [],
    'pedestrianTraffic': [], 'policeDist': [], 'shopCount': []
}
for _ in range(numRows):
    zone = random.choice(zones)
    time = random.choice(times)
    if zone == 'Commercial':
        lux = np.random.normal(80, 15)
        pedestrians = np.random.randint(50, 500)
        police = np.random.uniform(0.5, 2.0)
        shops = np.random.randint(5, 15)
    else:
        lux = np.random.normal(30, 20)
        pedestrians = np.random.randint(0, 100)
        police = np.random.uniform(2.0, 8.0)
        shops = np.random.randint(0, 4)
    data['zone'].append(zone)
    data['timeOfDay'].append(time)
    data['luxLevel'].append(max(0, lux))
    data['pedestrianTraffic'].append(pedestrians)
    data['policeDist'].append(police)
    data['shopCount'].append(shops)
df = pd.DataFrame(data)
dangerLambda = 1.0
dangerLambda += np.where(df['timeOfDay'] == 'LateNight', 2.5, 0)
dangerLambda += np.where((df['luxLevel'] < 20) & (df['pedestrianTraffic'] < 20), 3.0, 0)
dangerLambda -= np.where(df['shopCount'] > 5, 1.5, 0)
dangerLambda += (df['policeDist'] * 0.2)
dangerLambda = np.maximum(dangerLambda, 0.1)
df['incidents'] = np.random.poisson(dangerLambda)
def categorizeRisk(incidents):
    if incidents <= 1: return 'Safe'
    elif incidents <= 4: return 'Moderate'
    else: return 'High Risk'
df['riskClass'] = df['incidents'].apply(categorizeRisk)
df.drop('incidents', axis=1, inplace=True)
df.to_csv('cpted_safety_data.csv', index=False)
print(f"Generated {numRows} complex CPTED-backed records successfully.")

Generated 50000 complex CPTED-backed records successfully.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('cpted_safety_data.csv')
X = pd.get_dummies(df.drop('riskClass', axis=1))
y = df['riskClass']
XTrain, XTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
model.fit(XTrain, yTrain)
predictions = model.predict(XTest)
acc = accuracy_score(yTest, predictions)
f1 = f1_score(yTest, predictions, average='macro')
print(f"Model Accuracy: {acc * 100:.2f}%")
print(f"Model F1 Score: {f1 * 100:.2f}%\n")
print("Detailed Report:\n", classification_report(yTest, predictions))
def testCustomRoute():
    print("\n--- Test Your Own Route ---")
    inZone = input("Zone (Commercial/Residential/Industrial/Outskirts): ")
    inTime = input("Time (Morning/Afternoon/Evening/LateNight): ")
    inLux = float(input("Lux Level (0 to 100, e.g., 40): "))
    inPed = int(input("Pedestrian Traffic (e.g., 50): "))
    inPol = float(input("Police Distance in km (e.g., 2.5): "))
    inShop = int(input("Number of 24/7 Shops (e.g., 3): "))
    userInput = pd.DataFrame(columns=X.columns)
    userInput.loc[0] = 0
    userInput.at[0, 'luxLevel'] = inLux
    userInput.at[0, 'pedestrianTraffic'] = inPed
    userInput.at[0, 'policeDist'] = inPol
    userInput.at[0, 'shopCount'] = inShop
    zoneCol = f'zone_{inZone}'
    timeCol = f'timeOfDay_{inTime}'
    if zoneCol in userInput.columns: userInput.at[0, zoneCol] = 1
    if timeCol in userInput.columns: userInput.at[0, timeCol] = 1
    pred = model.predict(userInput)[0]
    print(f"\n>> The AI predicts this route is: {pred.upper()}")
# testCustomRoute()

Model Accuracy: 61.41%
Model F1 Score: 58.66%

Detailed Report:
               precision    recall  f1-score   support

   High Risk       0.53      0.45      0.49      1466
    Moderate       0.54      0.71      0.62      4235
        Safe       0.77      0.58      0.66      4299

    accuracy                           0.61     10000
   macro avg       0.61      0.58      0.59     10000
weighted avg       0.64      0.61      0.61     10000



In [11]:
 %%writefile routing_engine.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <string>
#include <limits>
using namespace std;
struct Edge {
    int dest;
    string riskCategory;
    double physicalDist;
};
void calculateSafestRoute(int startNode, int endNode, int numNodes, const vector<vector<Edge>>& cityGraph) {
    vector<double> minCost(numNodes, numeric_limits<double>::infinity());
    vector<int> parent(numNodes, -1);
    priority_queue<pair<double, int>, vector<pair<double, int>>, greater<pair<double, int>>> pq;
    minCost[startNode] = 0.0;
    pq.push({0.0, startNode});
    while (!pq.empty()) {
        double currentCost = pq.top().first;
        int currentNode = pq.top().second;
        pq.pop();
        if (currentNode == endNode) {
            break; // Stop if we reached the destination
        }
        if (currentCost > minCost[currentNode]) {
            continue; // Skip outdated paths in the queue
        }
        for (auto& road : cityGraph[currentNode]) {
            int nextNode = road.dest;
            double dangerMultiplier = 1.0;
            if (road.riskCategory == "Moderate") {
                dangerMultiplier = 2.5;
            } else if (road.riskCategory == "High Risk") {
                dangerMultiplier = 6.0; // Heavily penalize high risk streets
            }
            double roadCost = road.physicalDist * dangerMultiplier;
            double totalPathCost = currentCost + roadCost;
            if (totalPathCost < minCost[nextNode]) {
                minCost[nextNode] = totalPathCost;
                parent[nextNode] = currentNode;
                pq.push({totalPathCost, nextNode});
            }
        }
    }
    if (minCost[endNode] == numeric_limits<double>::infinity()) {
        cout << "Error: No safe path found to destination." << endl;
    } else {
        cout << ">> Pathfinding execution complete." << endl;
        cout << ">> Safest Route Danger-Adjusted Cost: " << minCost[endNode] << endl;
    }
}
int main() {
    int numNodes = 5;
    vector<vector<Edge>> cityGraph(numNodes);
    cityGraph[0].push_back({1, "Moderate", 2.0});
    cout << "--- Booting Optimal Safety Routing Engine ---" << endl;
    calculateSafestRoute(0, 1, numNodes, cityGraph);
    return 0;
}

Overwriting routing_engine.cpp


In [12]:
!g++ routing_engine.cpp -o routing_engine
!./routing_engine

--- Booting Optimal Safety Routing Engine ---
>> Pathfinding execution complete.
>> Safest Route Danger-Adjusted Cost: 5
